# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smartanilmali234-art/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I have selected the **SEO & Content Refresh Optimization (Freestyle)** lane. In digital marketing and content management, keeping articles fresh is essential for maintaining organic search traffic. However, manually auditing thousands of pages to decide which ones to rewrite is a massive operational bottleneck. By framing this as a predictive ML problem, we can build a decision-support tool that flags decaying content (pages losing impressions or traffic) and prioritizes which pages will yield the highest return on investment (ROI) if refreshed.

In [1]:
print("The anonymized starter CSV is loaded in Section 3.")

The anonymized starter CSV is loaded in Section 3.


## 2. The question: decision, action, cost of a wrong call

* **What decision does your work improve?** It improves the decision of *which* specific content pieces to assign to writers for a rewrite/refresh, and which ones to leave alone.
* **Who acts on it?** Content marketing managers, SEO strategists, and freelance writing teams.
* **What does a wrong recommendation cost?** * A **False Positive** (predicting a page needs a refresh when it doesn't) costs unnecessary writing budget and writer hours spent on healthy content.
  * A **False Negative** (missing a decaying page that desperately needs a refresh) is highly asymmetric. It leads to a catastrophic, compounding drop in search engine rankings, organic traffic, leads, and company revenue that takes months to recover.

In [2]:
import pandas as pd
import numpy as np

# Illustrative workflow weights, not measured financial costs.
cost_FP = 1   # An unnecessary editor review or content refresh
cost_FN = 4   # A declining page missed by the review queue

# 2. Represent this as a cost-loss penalty ratio
penalty_ratio = cost_FN / cost_FP

print("--- Operational Risk Profile ---")
print(f"False-positive workflow weight: {cost_FP}")
print(f"False-negative workflow weight: {cost_FN}")
print(f"Asymmetric Penalty Ratio (FN/FP): {penalty_ratio:.1f}x")
print("--------------------------------")

print("\nInterpretation: missing a genuinely declining page can delay review, while a false positive uses editor time.")

--- Operational Risk Profile ---
False-positive workflow weight: 1
False-negative workflow weight: 4
Asymmetric Penalty Ratio (FN/FP): 4.0x
--------------------------------

Interpretation: missing a genuinely declining page can delay review, while a false positive uses editor time.


In [3]:
print("This analysis uses repository-relative paths so it can be rerun after cloning the project.")

This analysis uses repository-relative paths so it can be rerun after cloning the project.


## 3. Quick look at the data (2-3 real numbers)

After successfully connecting to the `content_refresh_anonymized.csv` dataset, we observe the following baseline parameters:
* **Feature Scope:** The dataset contains **44 distinct features** detailing metrics like `search_volume`, `impressions_90d`, `days_since_last_update`, and traffic trends (`trend_direction`).
* **Volume:** The starter dataset contains **30,000 rows** for initial exploration.
* **Target Signals:** `trend_direction`, `ctr` (click-through rate), and `days_since_last_update` are relevant observed fields for later analysis; their usefulness must be tested rather than assumed.

In [4]:
import os
import glob
import pandas as pd

# 1. Point directly to your data path
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
data_dir = os.path.join(project_root, "data", "raw")
target_file = os.path.join(data_dir, "content_refresh_anonymized.csv")

# 2. Load the full dataset to get exact numbers for your markdown
df = pd.read_csv(target_file)
total_rows = len(df)
total_cols = len(df.columns)

print(f"Dataset Name: {os.path.basename(target_file)}")
print(f"Exact Total Rows: {total_rows:,}")
print(f"Exact Total Columns: {total_cols}")
print("\nUnique values in trend_direction (potential targets):")
print(df['trend_direction'].value_counts(dropna=False))

Dataset Name: content_refresh_anonymized.csv
Exact Total Rows: 30,000
Exact Total Columns: 44

Unique values in trend_direction (potential targets):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

* **What this work CAN claim:** This work provides **observed**, **directional**, and **decision-support** value. It uses historical search-performance signals to prioritize pages with an observed declining trend that may warrant editorial review.
* **What this work NEVER will claim:** This work will never claim **causal proof** (e.g., we cannot prove that *not* updating a page causes search engines to penalize it, as algorithmic updates are proprietary). It cannot guarantee traffic returns post-refresh, nor can it anticipate unpredictable macroeconomic traffic shifts or competitors publishing superior content overnight.

In [5]:
# Let's check missingness on this real SEO dataset
print("Initiating signal integrity check matrix...")
missing_values = df.isnull().sum()
columns_with_missing = missing_values[missing_values > 0]

if not columns_with_missing.empty:
    print("\nColumns with missing values and their counts:")
    print(columns_with_missing)
else:
    print("\nNo missing values found in the target columns!")

Initiating signal integrity check matrix...

Columns with missing values and their counts:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [6]:
import pandas as pd
import numpy as np

print("==================================================")
print("🔍 RUNNING SELF-CHECK: DATA INTEGRITY PIPELINE")
print("==================================================\n")

# Assert 1: DataFrame check
try:
    assert 'df' in globals() or 'sample_df' in globals(), "❌ Fail: No dataframe found in memory. Please run Section 3 first."
    working_df = df if 'df' in globals() else sample_df
    print(f"✅ Pass: Dataframe detected. Shape: {working_df.shape[0]} rows x {working_df.shape[1]} columns")
except AssertionError as e:
    print(e)
    working_df = None

if working_df is not None:
    # Assert 2: Essential columns check for SEO/refresh problem space
    critical_columns = ['content_id', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'trend_direction']
    missing_cols = [col for col in critical_columns if col not in working_df.columns]
    
    try:
        assert len(missing_cols) == 0, f"❌ Fail: Missing critical feature columns: {missing_cols}"
        print("✅ Pass: All core operational features are present in the dataset.")
    except AssertionError as e:
        print(e)

    # Assert 3: Missing Target Column Integrity
    try:
        # Checking if our primary target signal candidate has completely null fields
        null_target_count = working_df['trend_direction'].isnull().sum()
        target_null_percentage = (null_target_count / len(working_df)) * 100
        assert target_null_percentage < 5.0, f"❌ Fail: High rate of missingness in potential target 'trend_direction' ({target_null_percentage:.2f}%)"
        print(f"✅ Pass: 'trend_direction' has highly stable data (only {target_null_percentage:.2f}% nulls).")
    except AssertionError as e:
        print(e)

    # Assert 4: Data Types Verification
    try:
        # Check that we have numeric traffic/ranking columns to train with
        numeric_cols = working_df.select_dtypes(include=[np.number]).columns.tolist()
        assert len(numeric_cols) > 5, "❌ Fail: Dataset contains too few numeric signals."
        print(f"✅ Pass: Found {len(numeric_cols)} numerical signals to leverage for feature engineering.")
    except AssertionError as e:
        print(e)

print("\n==================================================")
print("🎉 SELF-CHECK COMPLETE: Ready to frame ML target tasks in ML-03!")
print("==================================================")

🔍 RUNNING SELF-CHECK: DATA INTEGRITY PIPELINE

✅ Pass: Dataframe detected. Shape: 30000 rows x 44 columns
✅ Pass: All core operational features are present in the dataset.
✅ Pass: 'trend_direction' has highly stable data (only 0.00% nulls).
✅ Pass: Found 30 numerical signals to leverage for feature engineering.

🎉 SELF-CHECK COMPLETE: Ready to frame ML target tasks in ML-03!
